# 🚀 High-Speed EVA-02 9-Tile Keyframe Batch Encoder (Colab G4 / GPU)
Encodes 10,000–50,000 video contact sheets into 9-tile MIL embeddings in minutes using Google Colab G4 (Blackwell RTX Pro 6000) or any high-VRAM GPU (A100 / L4 / T4).

### Instructions:
1. **Runtime Type**: In Colab menu, choose **Runtime -> Change runtime type -> G4 GPU (or A100 / L4 / T4)**.
2. Place your `colab_batch.tar.gz` archive in your Google Drive root or upload directly to `/content/` via the left file browser.
3. Run the cells below.

In [ ]:
# Step 1: Install Dependencies & Check GPU
!nvidia-smi
!pip install -q open-clip-torch timm torchvision pillow tqdm

In [ ]:
# Step 2: Locate Archive & Extract to High-Speed NVMe
import os, sys, tarfile, shutil

# Attempt to mount Google Drive if available
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
    print("[+] Google Drive mounted successfully.")
except Exception as e:
    print(f"[*] Note: Drive mount skipped or not interactive ({e}). Continuing with local storage...")

# Search candidate paths for the batch archive
CANDIDATE_PATHS = [
    '/content/drive/MyDrive/colab_batch.tar.gz',
    '/content/colab_batch.tar.gz',
    'colab_batch.tar.gz'
]

TAR_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        TAR_PATH = p
        break

if TAR_PATH is None:
    raise FileNotFoundError(
        "Could not find colab_batch.tar.gz! "
        "Please upload it to /content/ via the left file browser, or copy it to Google Drive root."
    )

tar_mb = os.path.getsize(TAR_PATH) / (1024 * 1024)
print(f"[*] Found archive: {TAR_PATH} ({tar_mb:.1f} MB)")
print("[*] Extracting to high-speed NVMe scratchpad (/content/work)...")
os.makedirs('/content/work', exist_ok=True)
with tarfile.open(TAR_PATH, 'r:gz') as tar:
    tar.extractall('/content/work')

print("[*] Extraction complete. Workspace contents:", os.listdir('/content/work'))

In [ ]:
# Step 3: High-Throughput 9-Tile Batch Encoder on Tensor Cores
import json, time, os, shutil
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.io import decode_jpeg, ImageReadMode
import open_clip

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

# Locate manifest and mosaics folder
manifest_file = '/content/work/manifest.json'
mosaics_dir = '/content/work/mosaics'
if not os.path.exists(manifest_file):
    for root, dirs, files in os.walk('/content/work'):
        if 'manifest.json' in files:
            manifest_file = os.path.join(root, 'manifest.json')
            mosaics_dir = os.path.join(root, 'mosaics')
            break

with open(manifest_file, 'r', encoding='utf-8') as f:
    items = json.load(f)

print(f"[*] Loaded manifest with {len(items):,} items to encode on {DEVICE}.")

# Dynamic batch sizing based on available GPU VRAM
if DEVICE == 'cuda':
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    BATCH_SIZE = 36 if total_vram_gb >= 20.0 else (16 if total_vram_gb >= 10.0 else 8)
    print(f"[*] GPU: {gpu_name} ({total_vram_gb:.1f} GB VRAM) -> Batch Size: {BATCH_SIZE} videos ({BATCH_SIZE * 9} tiles/step)")
else:
    BATCH_SIZE = 2
    print(f"[*] Running on CPU -> Batch Size: {BATCH_SIZE}")

class ColabMosaicDataset(Dataset):
    def __init__(self, item_list):
        self.items = item_list
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        entry = self.items[idx]
        img_path = os.path.join(mosaics_dir, entry['filename'])
        try:
            with open(img_path, 'rb') as f:
                raw_bytes = bytearray(f.read())
            raw_tensor = torch.frombuffer(raw_bytes, dtype=torch.uint8)
            im = decode_jpeg(raw_tensor, mode=ImageReadMode.RGB)
            _, h, w = im.shape
            tw, th = w // 3, h // 3
            crops = torch.stack([
                im[:, r*th:(r+1)*th, c*tw:(c+1)*tw] for r in range(3) for c in range(3)
            ], dim=0)
            resized = F.interpolate(crops.float(), size=(336, 336), mode='bilinear', align_corners=False).to(torch.uint8)
        except Exception:
            resized = torch.zeros(9, 3, 336, 336, dtype=torch.uint8)
        return resized, idx

def collate_crops(batch):
    tensors, indices = zip(*batch)
    return torch.cat(tensors, dim=0), list(indices)

num_workers = 4 if os.cpu_count() and os.cpu_count() >= 4 else 2
dataset = ColabMosaicDataset(items)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, collate_fn=collate_crops, pin_memory=(DEVICE == 'cuda'), prefetch_factor=2)

# Load EVA-02 in Native FP16
print("[*] Loading EVA-02-CLIP-L-14-336 in pure FP16 on Tensor Cores...")
model, _, _ = open_clip.create_model_and_transforms('hf-hub:timm/eva02_large_patch14_clip_336.merged2b_s6b_b61k', device=DEVICE)
if DEVICE == 'cuda':
    model = model.half().eval()
else:
    model = model.eval()

# Check for optional fine-tuned weights
for wpath in ['/content/work/model.pt', '/content/drive/MyDrive/eva02_curated_v2.pt']:
    if os.path.exists(wpath):
        print(f"[*] Loading custom fine-tuned weights from {wpath}...")
        ckpt = torch.load(wpath, map_location='cpu')
        state = ckpt.get('state_dict', ckpt)
        model.load_state_dict(state, strict=False)
        break

IMAGE_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=DEVICE, dtype=torch.float16 if DEVICE == 'cuda' else torch.float32).view(1, 3, 1, 1)
IMAGE_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=DEVICE, dtype=torch.float16 if DEVICE == 'cuda' else torch.float32).view(1, 3, 1, 1)

all_9tile_embeddings = []
all_1tile_embeddings = []
processed_indices = []

t0 = time.time()
encoded_count = 0

for flat_uint8, batch_indices in loader:
    B = len(batch_indices)
    flat_tensor = flat_uint8.to(DEVICE, non_blocking=True)
    if DEVICE == 'cuda':
        flat_tensor = flat_tensor.half()
    else:
        flat_tensor = flat_tensor.float()
    flat_norm = (flat_tensor / 255.0).sub_(IMAGE_MEAN).div_(IMAGE_STD)

    with torch.inference_mode():
        feats = model.encode_image(flat_norm)
        feats = F.normalize(feats.float(), dim=-1).half()
        tiles_9 = feats.view(B, 9, 768).cpu().numpy()
        tiles_1 = tiles_9[:, 4, :].copy()

    all_9tile_embeddings.append(tiles_9)
    all_1tile_embeddings.append(tiles_1)
    processed_indices.extend(batch_indices)
    encoded_count += B

    if encoded_count % (BATCH_SIZE * 4) == 0 or encoded_count == len(items):
        elapsed = max(0.1, time.time() - t0)
        rate = encoded_count / elapsed
        print(f"[{encoded_count:,} / {len(items):,}] Speed: {rate:.1f} vids/s ({rate*9:.1f} frames/s)")

final_9tiles = np.concatenate(all_9tile_embeddings, axis=0)
final_1tile = np.concatenate(all_1tile_embeddings, axis=0)

print(f"\n[+] Finished encoding! 9-Tiles Shape: {final_9tiles.shape}, Globals Shape: {final_1tile.shape}")
np.save('/content/encoded_keyframes_9tiles.npy', final_9tiles)
np.save('/content/encoded_globals.npy', final_1tile)

# Export matching manifest
encoded_meta = [items[i] for i in processed_indices]
with open('/content/encoded_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(encoded_meta, f, indent=2)

# Copy output to Drive if mounted, otherwise notify for local download
drive_target = '/content/drive/MyDrive'
if os.path.exists(drive_target):
    shutil.copy('/content/encoded_keyframes_9tiles.npy', os.path.join(drive_target, 'encoded_keyframes_9tiles.npy'))
    shutil.copy('/content/encoded_globals.npy', os.path.join(drive_target, 'encoded_globals.npy'))
    shutil.copy('/content/encoded_manifest.json', os.path.join(drive_target, 'encoded_manifest.json'))
    print(f"[+] Copied result files directly to your Google Drive root ({drive_target})!")
else:
    print("[+] Files saved locally in /content/:")
    print("    - /content/encoded_keyframes_9tiles.npy")
    print("    - /content/encoded_globals.npy")
    print("    - /content/encoded_manifest.json")
    print("    (You can download them directly from the Colab left file browser)")